In [37]:
import json

import geopandas as gpd
import gradio as gr
import h3
import numpy as np
import pandas as pd
import plotly.express as px
from shapely.geometry import shape

In [38]:
# 1. Load your CSV data
# Replace 'your_data.csv' and 'polygon_column' with your actual file and column names
df = pd.read_csv("nuremberg_2020_20m_polygons.csv")

# 2. Parse the JSON strings into Shapely geometry objects
# This reads that long string you pasted and turns it into a mathematical polygon
df["geometry"] = df[".geo"].apply(lambda x: shape(json.loads(x)))

# 3. Create a GeoDataFrame and explicitly tell it the current format is EPSG:3857 (meters)
gdf = gpd.GeoDataFrame(df, geometry="geometry", crs="EPSG:32632")

In [6]:
# 1. Load your CSV data
# Replace 'your_data.csv' and 'polygon_column' with your actual file and column names
df = pd.read_csv("nuremberg_2020_20m_polygons.csv")

# 2. Parse the JSON strings into Shapely geometry objects
# This reads that long string you pasted and turns it into a mathematical polygon
df["geometry"] = df[".geo"].apply(lambda x: shape(json.loads(x)))

In [7]:
df.columns

Index(['system:index', 'B11', 'B12', 'B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8',
       'bare_sparse_vegetation', 'built_up', 'cell_id', 'cropland',
       'grassland', 'image_count', 'tree_cover', 'water', 'x', 'y', 'year',
       '.geo', 'geometry'],
      dtype='str')

In [39]:
# Build a 3x3 neighborhood feature table (keep all original columns + add 8-neighbor bands)
import re


def build_3x3_neighborhood_df(
    df: pd.DataFrame,
    band_cols: list[str],
    cell_size: float = 20.0,
    group_cols: list[str] | None = None,
) -> pd.DataFrame:
    """
    Returns one row per center cell with all original columns plus neighbor-band features.

    - Keeps every original column from df unchanged.
    - Adds only the 8 surrounding neighbor band sets as new columns.
    - Center-cell bands remain original names (B11, B12, ...), no duplicated p0_p0 columns.
    - Uses inner joins for neighbors, so edge/missing-neighbor cells are dropped.
    """
    if group_cols is None:
        group_cols = [c for c in ["year"] if c in df.columns]

    required_cols = set(["x", "y", *band_cols, *group_cols])
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    merge_keys = [*group_cols, "x", "y"]

    # Keep all original columns as the base output
    out = df.copy()

    def offset_suffix(dx: int, dy: int) -> str:
        sx = f"{'m' if dx < 0 else 'p'}{abs(dx)}"
        sy = f"{'m' if dy < 0 else 'p'}{abs(dy)}"
        return f"{sx}_{sy}"

    for dy in (-1, 0, 1):
        for dx in (-1, 0, 1):
            # Skip center: those band columns already exist in out
            if dx == 0 and dy == 0:
                continue

            neigh = df[[*group_cols, "x", "y", *band_cols]].copy()

            # Shift neighbor coordinates into center coordinate frame
            neigh["x"] = neigh["x"] - dx * cell_size
            neigh["y"] = neigh["y"] - dy * cell_size

            suff = offset_suffix(dx, dy)
            neigh = neigh.rename(columns={c: f"{c}_{suff}" for c in band_cols})

            out = out.merge(neigh, on=merge_keys, how="inner")

    return out


# Example: infer band and output columns from your current schema
band_columns = [c for c in df.columns if re.fullmatch(r"B\d+", c)]

df_3x3 = build_3x3_neighborhood_df(
    df=df,
    band_cols=band_columns,
    cell_size=20.0,
    group_cols=["year"] if "year" in df.columns else None,
)

print("Original rows:", len(df))
print("3x3 valid center rows:", len(df_3x3))
print("Row diff:", len(df) - len(df_3x3))
print("Feature columns:", len(df_3x3.columns))
print(*df.columns)
print(*df_3x3.columns)
df_3x3.head(3)

Original rows: 466548
3x3 valid center rows: 457363
Row diff: 9185
Feature columns: 95
system:index B11 B12 B2 B3 B4 B5 B6 B7 B8 bare_sparse_vegetation built_up cell_id cropland grassland image_count tree_cover water x y year .geo geometry
system:index B11 B12 B2 B3 B4 B5 B6 B7 B8 bare_sparse_vegetation built_up cell_id cropland grassland image_count tree_cover water x y year .geo geometry B11_m1_m1 B12_m1_m1 B2_m1_m1 B3_m1_m1 B4_m1_m1 B5_m1_m1 B6_m1_m1 B7_m1_m1 B8_m1_m1 B11_p0_m1 B12_p0_m1 B2_p0_m1 B3_p0_m1 B4_p0_m1 B5_p0_m1 B6_p0_m1 B7_p0_m1 B8_p0_m1 B11_p1_m1 B12_p1_m1 B2_p1_m1 B3_p1_m1 B4_p1_m1 B5_p1_m1 B6_p1_m1 B7_p1_m1 B8_p1_m1 B11_m1_p0 B12_m1_p0 B2_m1_p0 B3_m1_p0 B4_m1_p0 B5_m1_p0 B6_m1_p0 B7_m1_p0 B8_m1_p0 B11_p1_p0 B12_p1_p0 B2_p1_p0 B3_p1_p0 B4_p1_p0 B5_p1_p0 B6_p1_p0 B7_p1_p0 B8_p1_p0 B11_m1_p1 B12_m1_p1 B2_m1_p1 B3_m1_p1 B4_m1_p1 B5_m1_p1 B6_m1_p1 B7_m1_p1 B8_m1_p1 B11_p0_p1 B12_p0_p1 B2_p0_p1 B3_p0_p1 B4_p0_p1 B5_p0_p1 B6_p0_p1 B7_p0_p1 B8_p0_p1 B11_p1_p1 B12_p1_p1 B2_p1_

,system:index,B11,B12,B2,B3,B4,B5,B6,B7,B8,...,B8_p0_p1,B11_p1_p1,B12_p1_p1,B2_p1_p1,B3_p1_p1,B4_p1_p1,B5_p1_p1,B6_p1_p1,B7_p1_p1,B8_p1_p1
0,18,1335.5,702.5,263.0,392.5,353.5,684.0,1532.0,1827.5,2063.0,...,1859.5,1314.5,687.5,248.5,381.5,314.5,645.5,1528.0,1828.5,1959.5
1,19,1429.5,749.5,253.0,410.5,363.5,692.5,1675.5,2039.5,2301.0,...,1959.5,1246.0,678.5,259.5,389.0,284.5,623.5,1517.0,1955.0,2048.0
2,20,1284.5,700.5,265.0,404.5,319.0,630.0,1469.0,1877.0,2068.0,...,2048.0,1057.5,543.5,233.5,329.0,258.0,533.0,1305.0,1515.5,1646.5


In [ ]:

# Load the CSV
df = pd.read_csv("nuremberg_2020_20m_polygons.csv")

# Parse the GeoJSON
df["geometry"] = df[".geo"].apply(lambda x: shape(json.loads(x)))

In [17]:
import json

import geopandas as gpd
import gradio as gr
import h3
import numpy as np
import pandas as pd
import plotly.express as px
from shapely.geometry import shape
from pathlib import Path
import re

def build_3x3_neighborhood_df(
    df: pd.DataFrame,
    band_cols: list[str],
    cell_size: float = 20.0,
    group_cols: list[str] | None = None,
) -> pd.DataFrame:
    """
    Returns one row per center cell with all original columns plus neighbor-band features.

    - Keeps every original column from df unchanged.
    - Adds only the 8 surrounding neighbor band sets as new columns.
    - Center-cell bands remain original names (B11, B12, ...), no duplicated p0_p0 columns.
    - Uses inner joins for neighbors, so edge/missing-neighbor cells are dropped.
    """
    if group_cols is None:
        group_cols = [c for c in ["year"] if c in df.columns]

    required_cols = set(["x", "y", *band_cols, *group_cols])
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    merge_keys = [*group_cols, "x", "y"]

    # Keep all original columns as the base output
    out = df.copy()

    def offset_suffix(dx: int, dy: int) -> str:
        sx = f"{'m' if dx < 0 else 'p'}{abs(dx)}"
        sy = f"{'m' if dy < 0 else 'p'}{abs(dy)}"
        return f"{sx}_{sy}"

    for dy in (-1, 0, 1):
        for dx in (-1, 0, 1):
            # Skip center: those band columns already exist in out
            if dx == 0 and dy == 0:
                continue

            neigh = df[[*group_cols, "x", "y", *band_cols]].copy()

            # Shift neighbor coordinates into center coordinate frame
            neigh["x"] = neigh["x"] - dx * cell_size
            neigh["y"] = neigh["y"] - dy * cell_size

            suff = offset_suffix(dx, dy)
            neigh = neigh.rename(columns={c: f"{c}_{suff}" for c in band_cols})

            out = out.merge(neigh, on=merge_keys, how="inner")

    return out



In [ ]:
data_path = "data"
original_files = list(Path(data_path).iterdir())
output_files = [Path(str(og.parent.absolute() / og.stem) + "_3x3" + str(og.suffix)) for og in original_files]

for in_file, out_file in zip(original_files, output_files):

    # Skip if the output file already exists
    if out_file.exists() and out_file.is_file():
        print(f"{out_file} already exists. Skipping...")
        continue

    print(f"Starting on input: {in_file.name}; output: {out_file.name}")
    # Load the CSV
    df = pd.read_csv(in_file, index_col=False)

    # Parse the GeoJSON
    df["geometry"] = df[".geo"].apply(lambda x: shape(json.loads(x)))

    # Example: infer band and output columns from your current schema
    band_columns = [c for c in df.columns if re.fullmatch(r"B\d+", c)]

    df_3x3 = build_3x3_neighborhood_df(
        df=df,
        band_cols=band_columns,
        cell_size=20.0,
        group_cols=["year"] if "year" in df.columns else None,
    )

    print("Original rows:", len(df))
    print("3x3 valid center rows:", len(df_3x3))
    print("Row diff:", len(df) - len(df_3x3))
    print("Feature columns:", len(df_3x3.columns))
    print(*df.columns)
    print(*df_3x3.columns)

    print(f"Saving {out_file}...")
    df_3x3.to_csv(out_file, index=False)

Starting on input: df_2016_mit_2021_labels.csv; output: datadf_2016_mit_2021_labels_3x3.csv
Original rows: 466548
3x3 valid center rows: 457363
Row diff: 9185
Feature columns: 95
system:index B11 B12 B2 B3 B4 B5 B6 B7 B8 bare_sparse_vegetation built_up cropland grassland image_count tree_cover water x y year .geo class_label geometry
system:index B11 B12 B2 B3 B4 B5 B6 B7 B8 bare_sparse_vegetation built_up cropland grassland image_count tree_cover water x y year .geo class_label geometry B11_m1_m1 B12_m1_m1 B2_m1_m1 B3_m1_m1 B4_m1_m1 B5_m1_m1 B6_m1_m1 B7_m1_m1 B8_m1_m1 B11_p0_m1 B12_p0_m1 B2_p0_m1 B3_p0_m1 B4_p0_m1 B5_p0_m1 B6_p0_m1 B7_p0_m1 B8_p0_m1 B11_p1_m1 B12_p1_m1 B2_p1_m1 B3_p1_m1 B4_p1_m1 B5_p1_m1 B6_p1_m1 B7_p1_m1 B8_p1_m1 B11_m1_p0 B12_m1_p0 B2_m1_p0 B3_m1_p0 B4_m1_p0 B5_m1_p0 B6_m1_p0 B7_m1_p0 B8_m1_p0 B11_p1_p0 B12_p1_p0 B2_p1_p0 B3_p1_p0 B4_p1_p0 B5_p1_p0 B6_p1_p0 B7_p1_p0 B8_p1_p0 B11_m1_p1 B12_m1_p1 B2_m1_p1 B3_m1_p1 B4_m1_p1 B5_m1_p1 B6_m1_p1 B7_m1_p1 B8_m1_p1 B11_p0_p1

KeyboardInterrupt: 

In [23]:
Path("./33.zip").absolute().stem

'33'

In [40]:
from pathlib import Path

out_path = Path("df_3x3.csv")
df_3x3.to_csv(out_path, index=False)

print(f"Saved: {out_path.resolve()}")
print(f"Rows: {len(df_3x3):,} | Cols: {df_3x3.shape[1]:,}")

Saved: /home/macedmo/git/machine-learning-final-project-ws2025/dashboard/df_3x3.csv
Rows: 457,363 | Cols: 95


In [4]:
import pandas as pd
import geopandas as gpd
import osmnx as ox
from shapely.geometry import Point
from shapely import wkt

import json
from shapely.geometry import shape

# Updated parsing function
def parse_geometry(geo_str):
    if pd.isna(geo_str) or not isinstance(geo_str, str):
        return None
    try:
        # 1. Parse the string into a Python dictionary
        geo_dict = json.loads(geo_str)
        # 2. Convert that dictionary into a Shapely object
        return shape(geo_dict)
    except:
        return None

# 1. Load your data
df = pd.read_csv("data.csv") # Assuming your sample is in a CSV
df['geometry'] = df['.geo'].apply(parse_geometry)
gdf = gpd.GeoDataFrame(df, geometry='geometry', crs="EPSG:32632")

# 2. Define your target area (Nuremberg) for OSM data
place_name = "Nuremberg, Germany"

# 3. Fetch Road and Water data from OpenStreetMap
# Fetching "primary" and "secondary" roads as they are major change drivers
roads = ox.features_from_place(place_name, tags={"highway": ["primary", "secondary", "tertiary"]})
water = ox.features_from_place(place_name, tags={"natural": "water", "waterway": True})

# 4. Project OSM data to match your UTM 32N (EPSG:32632) for meter-based distance
roads = roads.to_crs("EPSG:32632")
water = water.to_crs("EPSG:32632")

# 5. Create a "Built-up" reference from your OWN 2016 data
# This is crucial for predicting change based on existing urban footprints [cite: 104]
existing_urban = gdf[gdf['built_up'] > 0.5].unary_union

def get_min_dist(point, target_gdf):
    """Calculates minimum distance from a point to any object in a GeoDataFrame."""
    return target_gdf.distance(point).min()

# 6. Calculate proximity features
print("Calculating distances... this may take a moment.")
gdf['dist_to_road'] = gdf.geometry.centroid.apply(lambda x: get_min_dist(x, roads))
gdf['dist_to_water'] = gdf.geometry.centroid.apply(lambda x: get_min_dist(x, water))
gdf['dist_to_builtup'] = gdf.geometry.centroid.apply(lambda x: x.distance(existing_urban))

print(gdf[['dist_to_road', 'dist_to_water', 'dist_to_builtup']].head())

/tmp/ipykernel_293076/3264745948.py:41: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  existing_urban = gdf[gdf['built_up'] > 0.5].unary_union


Calculating distances... this may take a moment.
   dist_to_road  dist_to_water  dist_to_builtup
0    349.033473      74.740731       496.588361
1    360.386728      93.039641       505.371151
2    372.468552     111.922174       514.781507
3    385.210399     131.136460       524.785671
4    398.548968     150.555535       535.350353


In [5]:
gdf.to_csv("data_with_extras.csv")